In [ ]:
import os, sys
import numpy as np
import pandas as pd

# ensure scikit-learn available
try:
    import sklearn
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.inspection import permutation_importance
    from sklearn.metrics import r2_score
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'scikit-learn'])
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.inspection import permutation_importance
    from sklearn.metrics import r2_score

infile = '/Users/mel-neiquaholloway/Downloads/gym_members_loaded.csv'
if not os.path.exists(infile):
    print('File not found:', infile); raise SystemExit(1)

df = pd.read_csv(infile)
print('Loaded:', df.shape)
if 'Calories_Burned' not in df.columns:
    print('No Calories_Burned column found'); raise SystemExit(1)

# Clean and numeric conversions
df = df.copy()
for col in ['Max_BPM','Avg_BPM','Resting_BPM']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Define feature groups
bio_cols = [c for c in ['Age','Weight (kg)','Height (m)','Fat_Percentage','BMI','Resting_BPM','Max_BPM'] if c in df.columns]
habit_cols = [c for c in ['Session_Duration (hours)','Workout_Frequency (days/week)','Experience_Level','Avg_BPM','Water_Intake (liters)'] if c in df.columns]
cat_cols = [c for c in ['Gender','Workout_Type'] if c in df.columns]
print('Bio cols:', bio_cols)
print('Habit cols:', habit_cols)
print('Cat cols:', cat_cols)

# Drop rows missing target
df = df[df['Calories_Burned'].notna()].reset_index(drop=True)
y = df['Calories_Burned'].values

# Fit encoder robustly and get dense matrix
if cat_cols:
    enc = OneHotEncoder(handle_unknown='ignore')
    cat_mat = enc.fit_transform(df[cat_cols].astype(str))
    # convert sparse matrix to dense if needed
    try:
        cat_mat = cat_mat.toarray()
    except Exception:
        pass
    cat_names = list(enc.get_feature_names_out(cat_cols))
else:
    cat_mat = None
    cat_names = []

# helper to build X
def build_X(cols, include_cat=True):
    parts = []
    names = []
    if cols:
        arr = df[cols].apply(pd.to_numeric, errors='coerce')
        # fill numeric NaNs with median of each column
        arr = arr.fillna(arr.median())
        parts.append(arr.values)
        names += cols
    if include_cat and cat_mat is not None:
        parts.append(cat_mat)
        names += cat_names
    if parts:
        X = np.hstack(parts)
    else:
        X = np.zeros((len(df),0))
    return X, names

X_bio_cat, names_bio_cat = build_X(bio_cols, include_cat=True)
X_habit_cat, names_habit_cat = build_X(habit_cols, include_cat=True)
X_all, names_all = build_X(bio_cols+habit_cols, include_cat=True)

rng = 42
results = {}
for label, X, names in [('bio', X_bio_cat, names_bio_cat), ('habit', X_habit_cat, names_habit_cat), ('all', X_all, names_all)]:
    if X.shape[1]==0:
        results[label] = {'r2': None, 'importances': []}
        continue
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rng)
    model = RandomForestRegressor(n_estimators=200, random_state=rng)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=rng)
    importances = sorted(zip(names, perm.importances_mean), key=lambda x: x[1], reverse=True)
    results[label] = {'r2': r2, 'importances': importances[:10]}

# Output concise results
print('\nR2 scores:')
for k in ['bio','habit','all']:
    print(f"  {k}: {results[k]['r2']:.3f}" if results[k]['r2'] is not None else f"  {k}: no features")

print('\nTop importances per group:')
for k in ['bio','habit','all']:
    print(f"\nGroup: {k}")
    for name,imp in results[k]['importances']:
        print(f"  {name}: {imp:.4f}")

# Simple correlation check
num_feats = [c for c in df.select_dtypes(include=[np.number]).columns if c!='Calories_Burned']
corrs = df[num_feats].corrwith(df['Calories_Burned']).abs().sort_values(ascending=False)
print('\nTop numeric correlations:')
print(corrs.head(10))

# Save summary
out = '/Users/mel-neiquaholloway/Downloads/gym_calorie_analysis_summary.txt'
with open(out,'w') as f:
    for k in ['bio','habit','all']:
        f.write(f"{k}: r2={results[k]['r2']}\n")
    f.write('\nTop importances:\n')
    for k in ['bio','habit','all']:
        f.write(f'Group: {k}\n')
        for name,imp in results[k]['importances']:
            f.write(f"{name}: {imp}\n")
    f.write('\nTop correlations:\n')
    f.write(str(corrs.head(10).to_dict()))
print('\nWrote summary to', out)
PY

Loaded: (1800, 15)
